# 03 — Warehouse Build (Phase 3)

โน้ตบุ๊กนี้เป็น narrative shell สำหรับ Phase 3: import และเรียกใช้ฟังก์ชันจาก `src/rri/warehouse/`
เท่านั้น — ไม่มี business logic ในเซลล์ตาม CLAUDE.md N2

ลำดับขั้นตอน (ตาม CLAUDE.md §8):

1. โหลดข้อมูล staging (จาก Phase 1-2, `data/10_staging/`)
2. สร้าง 8 dimension tables (มี Unknown Member `sk = -1` ทุกตาราง ตาม N7)
3. สร้าง `fact_accident` (grain: หนึ่งอุบัติเหตุ)
4. เขียนลง `warehouse.duckdb`
5. **CHECKPOINT** — reconciliation: Gate 2, orphan FK, grain uniqueness, Unknown Member usage


## 0. Setup

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import duckdb
import polars as pl

from rri.utils.logging import configure_logging
from rri.warehouse.build import build_dimensions, load_staging_frames
from rri.warehouse.dim_vehicle import vehicle_group_mapping_path
from rri.warehouse.duckdb_io import get_connection, write_table
from rri.warehouse.fact_accident import build_fact_accident
from rri.warehouse.keys import UNKNOWN_SK

configure_logging()

'5cac200a48e54663ac48c279ebc6fe42'

## 1. โหลดข้อมูล staging

อ่าน parquet ที่ผ่าน Phase 1-2 มาแล้ว (`data/10_staging/`) — ไม่มีการ clean ซ้ำในเฟสนี้

In [2]:
staging = load_staging_frames()
{name: df.height for name, df in staging.items()}

{'accidents': 159605,
 'flood_risk': 27023,
 'heavy_rain_r35mm': 3465,
 'stt_car_fuel': 4643,
 'car_registered_new': 969}

## 2. สร้าง 8 Dimension Tables

`build_dimensions` สร้างครบทั้ง 8 ตามที่ระบุใน CLAUDE.md §8 — `dim_policy_type`/`dim_customer_segment`
ยังเป็น scaffold (Unknown Member แถวเดียว) ในเฟสนี้ จะถูก populate จริงใน Phase 4 (notebook 04)

In [3]:
dims = build_dimensions(staging)
{name: df.height for name, df in dims.items()}

{'dim_date': 2923,
 'dim_geography': 78,
 'dim_vehicle': 6,
 'dim_accident_context': 1679,
 'dim_time_of_day': 25,
 'dim_weather_band': 6,
 'dim_policy_type': 5,
 'dim_customer_segment': 21}

### ตรวจสอบ Unknown Member (sk = -1)

ทุก dimension ต้องมีแถว Unknown Member ตาม N7 — ใช้เมื่อ fact row ชี้ไปยัง dimension value
ที่ไม่พบ (เช่น จังหวัดว่าง) แทนที่จะ drop แถวนั้นทิ้ง

In [4]:
for name, df in dims.items():
    has_unknown = df.filter(pl.col(df.columns[0]) == UNKNOWN_SK).height == 1
    print(f"{name}: {df.height} rows, Unknown Member present = {has_unknown}")

dim_date: 2923 rows, Unknown Member present = True
dim_geography: 78 rows, Unknown Member present = True
dim_vehicle: 6 rows, Unknown Member present = True
dim_accident_context: 1679 rows, Unknown Member present = True
dim_time_of_day: 25 rows, Unknown Member present = True
dim_weather_band: 6 rows, Unknown Member present = True
dim_policy_type: 5 rows, Unknown Member present = True
dim_customer_segment: 21 rows, Unknown Member present = True


## 3. สร้าง `fact_accident`

Grain: หนึ่งอุบัติเหตุ หนึ่งวันที่/เวลา หนึ่งจังหวัด หนึ่งช่วงถนน หนึ่งประเภทรถคันแรก
หนึ่งสาเหตุ และหนึ่งสภาพอากาศที่รายงาน — ทุก FK resolve ผ่านการ join กับ dimension ที่สร้างไว้แล้ว
ค่าที่ resolve ไม่ได้ (เช่น จังหวัดว่าง) จะชี้ไปที่ Unknown Member (`sk = -1`) แทนการ drop แถว (N7)

`agency` (หน่วยงาน) เป็น degenerate attribute บน fact โดยตรง ไม่ใช่ dimension แยก — ดู
CLAUDE.md §8 (แก้ไขความขัดแย้งเดิมระหว่าง FK list กับ Dimensions(8) enumeration แล้ว)

In [5]:
fact_accident = build_fact_accident(
    staging["accidents"],
    dims["dim_date"],
    dims["dim_geography"],
    dims["dim_vehicle"],
    dims["dim_accident_context"],
    dims["dim_time_of_day"],
    dims["dim_weather_band"],
    vehicle_group_mapping_path(),
)
fact_accident.height

159605

## 4. เขียนลง `warehouse.duckdb`

In [6]:
con = get_connection()
try:
    for table_name, df in dims.items():
        write_table(con, table_name, df)
    write_table(con, "fact_accident", fact_accident)
finally:
    con.close()
print("written:", [*dims.keys(), "fact_accident"])

2026-08-14T07:19:34.766985Z [info     ] wrote_table                    cols=21 rows=2923 run_id=5cac200a48e54663ac48c279ebc6fe42 table=dim_date


2026-08-14T07:19:34.769075Z [info     ] wrote_table                    cols=15 rows=78 run_id=5cac200a48e54663ac48c279ebc6fe42 table=dim_geography


2026-08-14T07:19:34.770328Z [info     ] wrote_table                    cols=4 rows=6 run_id=5cac200a48e54663ac48c279ebc6fe42 table=dim_vehicle


2026-08-14T07:19:34.772102Z [info     ] wrote_table                    cols=5 rows=1679 run_id=5cac200a48e54663ac48c279ebc6fe42 table=dim_accident_context


2026-08-14T07:19:34.773115Z [info     ] wrote_table                    cols=4 rows=25 run_id=5cac200a48e54663ac48c279ebc6fe42 table=dim_time_of_day


2026-08-14T07:19:34.773978Z [info     ] wrote_table                    cols=3 rows=6 run_id=5cac200a48e54663ac48c279ebc6fe42 table=dim_weather_band


2026-08-14T07:19:34.775002Z [info     ] wrote_table                    cols=4 rows=5 run_id=5cac200a48e54663ac48c279ebc6fe42 table=dim_policy_type


2026-08-14T07:19:34.775858Z [info     ] wrote_table                    cols=4 rows=21 run_id=5cac200a48e54663ac48c279ebc6fe42 table=dim_customer_segment


2026-08-14T07:19:34.887535Z [info     ] wrote_table                    cols=24 rows=159605 run_id=5cac200a48e54663ac48c279ebc6fe42 table=fact_accident


written: ['dim_date', 'dim_geography', 'dim_vehicle', 'dim_accident_context', 'dim_time_of_day', 'dim_weather_band', 'dim_policy_type', 'dim_customer_segment', 'fact_accident']


## 5. CHECKPOINT — Reconciliation

ตรวจ 5 อย่างตามที่ตกลงไว้ ก่อนไปต่อ Phase 4:
1. Gate 2 (post-load): fatalities == 21,497, rows == 159,605
2. Orphan check: fact row ที่ FK ไม่พบใน dimension ใดๆ ต้องเป็น 0
3. Grain uniqueness: `COUNT(*) == COUNT(DISTINCT business_key)`
4. Unknown Member usage: กี่แถวต่อ dimension และอธิบายได้ทุกกรณี
5. Row-count staging → fact: ครบทุกแถว ไม่มี exclusion ที่ไม่ได้ log ไว้

In [7]:
warehouse_path = Path.cwd().parent / "data" / "30_warehouse" / "warehouse.duckdb"
con = duckdb.connect(str(warehouse_path), read_only=True)

gate2 = con.sql(
    "SELECT COUNT(*) AS rows, SUM(fatality_count) AS fatalities FROM fact_accident"
).df()
print("Gate 2:")
print(gate2)
assert gate2["rows"][0] == 159_605
assert gate2["fatalities"][0] == 21_497

Gate 2:
     rows  fatalities
0  159605     21497.0


In [8]:
fks = {
    "date_sk": ("dim_date", "date_sk"),
    "geography_sk": ("dim_geography", "geography_sk"),
    "vehicle_sk": ("dim_vehicle", "vehicle_sk"),
    "accident_context_sk": ("dim_accident_context", "accident_context_sk"),
    "time_of_day_sk": ("dim_time_of_day", "time_of_day_sk"),
    "weather_band_sk": ("dim_weather_band", "weather_band_sk"),
}
print("Orphan FK check (must all be 0):")
total_orphans = 0
for fk, (dim, dimkey) in fks.items():
    n = con.sql(
        f"SELECT COUNT(*) FROM fact_accident f LEFT JOIN {dim} d ON f.{fk} = d.{dimkey} "
        f"WHERE d.{dimkey} IS NULL"
    ).fetchone()[0]
    total_orphans += n
    print(f"  {fk}: {n}")
assert total_orphans == 0

Orphan FK check (must all be 0):
  date_sk: 0
  geography_sk: 0
  vehicle_sk: 0
  accident_context_sk: 0
  time_of_day_sk: 0
  weather_band_sk: 0


In [9]:
grain = con.sql("""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT COALESCE(acc_code, 'ROW_' || accident_sk)) AS n_distinct_keys
    FROM fact_accident
""").df()
print("Grain uniqueness:")
print(grain)
assert grain["n_rows"][0] == grain["n_distinct_keys"][0]

Grain uniqueness:
   n_rows  n_distinct_keys
0  159605           159605


In [10]:
print("Unknown Member usage per dimension:")
for fk in fks:
    n = con.sql(f"SELECT COUNT(*) FROM fact_accident WHERE {fk} = -1").fetchone()[0]
    print(f"  {fk} = -1: {n} rows")
# geography (1,966) / vehicle (1,936) / weather (771) ล้วนตรงกับจำนวน raw source value
# ที่เป็น null พอดี (ตรวจสอบละเอียดใน Phase 3 checkpoint report) — ไม่มี join ที่ผิดพลาดแอบแฝง

Unknown Member usage per dimension:
  date_sk = -1: 1 rows
  geography_sk = -1: 1966 rows
  vehicle_sk = -1: 1936 rows
  accident_context_sk = -1: 0 rows
  time_of_day_sk = -1: 0 rows
  weather_band_sk = -1: 771 rows


**สรุป checkpoint:** Gate 2 ตรงเป๊ะ, orphan FK = 0, grain unique, Unknown Member usage
ทุกตัวอธิบายได้ตรงกับ null ของ raw source (ตรวจสอบ breakdown ละเอียดแล้วนอกโน้ตบุ๊กนี้)
→ พร้อมไปต่อ Phase 4

In [11]:
con.close()